In [5]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func
import torch.optim as optim
from torch.amp import GradScaler, autocast

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)

        self.initial = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

    def forward(self, x):
        x0 = self.initial(x)
        x1 = self.encoder1(x0)
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x1)

        out = self.final_conv(d1)
        return out

In [6]:
### Fourier Domain Adaptation

@torch.no_grad()
def fda(src_img: torch.Tensor, tgt_img: torch.Tensor, beta) -> torch.Tensor:
    C, H, W = src_img.shape

    src_fft = torch.fft.fft2(src_img, dim=(-2, -1))
    tgt_fft = torch.fft.fft2(tgt_img, dim=(-2, -1))

    src_amp, src_phase = torch.abs(src_fft), torch.angle(src_fft)
    tgt_amp = torch.abs(tgt_fft)

    src_amp_shift = torch.fft.fftshift(src_amp, dim=(-2, -1))
    tgt_amp_shift = torch.fft.fftshift(tgt_amp, dim=(-2, -1))

    b_h = int(beta * H)
    b_w = int(beta * W)
    c_h, c_w = H // 2, W // 2
    h1, h2 = c_h - b_h, c_h + b_h
    w1, w2 = c_w - b_w, c_w + b_w

    src_amp_shift[..., h1:h2, w1:w2] = tgt_amp_shift[..., h1:h2, w1:w2]

    src_amp_new = torch.fft.ifftshift(src_amp_shift, dim=(-2, -1))
    src_fft_new = src_amp_new * torch.exp(1j * src_phase)
    src_in_tgt = torch.fft.ifft2(src_fft_new, dim=(-2, -1)).real

    return torch.clamp(src_in_tgt, 0.0, 255.0)

In [ ]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

to_tensor = transforms.ToTensor()
normalize_05 = transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (to_tensor(mask) > 0).float()

        return img, mask

class SourceDataset(Dataset):
    def __init__(self, source_ds, target_ds, beta, normalizer=normalize_05):
        self.source = source_ds
        self.target = target_ds
        self.beta = beta
        self.normalizer = normalizer

    def __len__(self):
        return len(self.source)
    
    def _sample_target_img(self):
        id = random.randrange(len(self.target))
        tgt_path = self.target[id][random.randrange(len(self.target[id]))]

        img = tif.imread(tgt_path)
        img = to_tensor(img)
        return img
    
    def __getitem__(self, idx):
        src = self.source[idx]
        mask = src.replace("img", "mask")

        src_img = tif.imread(src)
        src_mask = tif.imread(mask)

        src_img = to_tensor(src_img)
        src_mask = (to_tensor(src_mask) > 0).float()

        tgt_img = self._sample_target_img()

        src255 = (src_img * 255.0).float().to(device, non_blocking=True)
        tgt255 = (tgt_img * 255.0).float().to(device, non_blocking=True)
        
        fda_src255 = fda(src255, tgt255, beta=self.beta)
        fda_src = fda_src255 / 255.0

        fda_src = self.normalizer(fda_src)

        return fda_src, src_mask


def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

path = "../data/"
regions = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]

subdataset_to_region = {
    "Hokkaido Iburi-Tobu": "Hokkaido Iburi-Tobu",
    "Jiuzhai valley (UAV-0.2m)": "Jiuzhai Valley",
    "Jiuzhai valley (UAV-0.5m)": "Jiuzhai Valley",
    "Lombok": "Lombok",
    "Longxi River (SAT)": "Longxi River",
    "Longxi River (UAV)": "Longxi River",
    "Mengdong Township": "Mengdong Township",
    "Moxi town (UAV-0.2m)": "Luding",
    "Moxi town (UAV-1m)": "Luding",
    "Moxitaidi (SAT)": "Luding",
    "Moxitaidi (UAV-0.6m)": "Luding",
    "Moxitaidi (UAV-1m)": "Luding",
    "palu": "Palu",
    "Tiburon Peninsula (planet)": "Tiburon Peninsula",
    "Tiburon Peninsula (Sentinel)": "Tiburon Peninsula",
    "Wenchuan": "Wenchuan"
}

regions_dict = {
    "Hokkaido Iburi-Tobu": [],
    "Jiuzhai Valley": [],
    "Lombok": [],
    "Longxi River": [],
    "Mengdong Township": [],
    "Luding": [],
    "Palu": [],
    "Tiburon Peninsula": [],
    "Wenchuan": []
}

regions_dict_subdataset = {
    "Hokkaido Iburi-Tobu": [],
    "Jiuzhai Valley": [],
    "Lombok": [],
    "Longxi River": [],
    "Mengdong Township": [],
    "Luding": [],
    "Palu": [],
    "Tiburon Peninsula": [],
    "Wenchuan": []
}

for region in regions:
    if(region != "study areas shp"):
        dataset_dir = "../data/" + region
        image_dir = os.path.join(dataset_dir, "img")
        img_list = os.listdir(image_dir)
        
        all_images = sorted(os.path.join(image_dir, f) for f in img_list)
        
        regions_dict[subdataset_to_region[region]].extend(all_images)
        regions_dict_subdataset[subdataset_to_region[region]].append(all_images)
        
for region in regions_dict:
    if(len(regions_dict[region]) > 1000):
        seededRandom = random.Random(42)
        regions_dict[region] = seededRandom.sample(regions_dict[region], 1000)

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

pos_weight = torch.tensor([4.5]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [10]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

batch_size = 16

beta = 0.05
beta_name = "01"

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in ["Hokkaido Iburi-Tobu"]:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            model = ResNetUNet().to(device).to(memory_format=torch.channels_last)

            target_images = regions_dict_subdataset[target_region]

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_images, beta)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            epochs = 40
    
            best_iou = 0.0
            patience = 10
            counter = 0

            save_path = f"../models/fda/{beta_name}/{beta_name}_{source_region}_{target_region}.pth"

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            scaler = GradScaler()

            print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')

            for epoch in range(epochs):
                model.train()
                running_loss = 0.0
                train_num = 0

                for i, data in enumerate(trainLoader, 0):
                    image, mask = data
                    
                    image = image.to(device)
                    mask = mask.to(device)
                    
                    image.to(memory_format=torch.channels_last)

                    optimizer.zero_grad()
                    
                    with autocast(device_type="cuda"):
                        outputs = model(image)
                        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                        bce = criterion(outputs, mask)
                        dice = dice_loss(outputs, mask)
                        loss = bce + dice
                    
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                    running_loss += loss.item()
                    train_num += 1
                
                model.eval()
                val_loss = 0.0
                val_num = 0
                
                TP, FP, FN, TN = 0,0,0,0
                
                with torch.no_grad():
                    for data in valLoader:
                        image, mask = data

                        image = image.to(device)
                        mask = mask.to(device)
                    
                        outputs = model(image)
                        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                        loss = criterion(outputs, mask)
                        val_loss += loss.item()

                        preds = torch.sigmoid(outputs) > .6
                        
                        preds_flat = preds.view(-1)
                        mask_flat = mask.view(-1)
                        
                        TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                        FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                        FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                        TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                        
                        val_num += 1
                
                precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
                
                print(f'{source_region}, {target_region}, {epoch+1}, {running_loss / train_num :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')
                
                if iou > best_iou:
                    best_iou = iou
                    counter = 0
                    
                    torch.save(model.state_dict(), save_path)
                elif iou != 0.0:
                    counter += 1
                    if counter >= patience:
                        break
            
            model.load_state_dict(torch.load(save_path, map_location=device))
            
            TP, FP, FN, TN = 0,0,0,0
            
            for data in testLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
                    
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

                preds = torch.sigmoid(outputs) > .6
                        
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                        
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
            precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
            output += f'\n{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'

with open(f"../results/fda/{beta_name}.csv", "w") as f:
    f.write(output)

region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, Jiuzhai Valley, 1, 1.697, 0.737, 0.0000, 0.0000, 0.0000, 0.0000, 0.4528, 0.9057
Hokkaido Iburi-Tobu, Jiuzhai Valley, 2, 1.475, 0.680, 0.5973, 0.2969, 0.3966, 0.2474, 0.5798, 0.9148
Hokkaido Iburi-Tobu, Jiuzhai Valley, 3, 1.299, 0.504, 0.5957, 0.4981, 0.5425, 0.3722, 0.6446, 0.9208
Hokkaido Iburi-Tobu, Jiuzhai Valley, 4, 1.207, 0.439, 0.5806, 0.6418, 0.6097, 0.4385, 0.6780, 0.9225
Hokkaido Iburi-Tobu, Jiuzhai Valley, 5, 1.143, 0.424, 0.4942, 0.7462, 0.5946, 0.4231, 0.6599, 0.9040
Hokkaido Iburi-Tobu, Jiuzhai Valley, 6, 1.102, 0.406, 0.4905, 0.7708, 0.5995, 0.4281, 0.6616, 0.9028
Hokkaido Iburi-Tobu, Jiuzhai Valley, 7, 1.113, 0.494, 0.5929, 0.5873, 0.5901, 0.4185, 0.6685, 0.9230
Hokkaido Iburi-Tobu, Jiuzhai Valley, 8, 1.040, 0.422, 0.4589, 0.7833, 0.5788, 0.4072, 0.6455, 0.8924
Hokkaido Iburi-Tobu, Jiuzhai Valley, 9, 1.043, 0.386, 0.5486, 0.7640, 0.6386, 0.4691, 0.6906, 0.9184
Hokkaido Iburi-To